In [ ]:
"""Quickstart example using Prophet backend."""

import pandas as pd
import numpy as np
from universal_ts import UniversalForecaster, evaluate

# Create synthetic daily data
np.random.seed(42)
dates = pd.date_range('2020-01-01', periods=365, freq='D')
trend = np.arange(365) * 0.1
seasonality = 10 * np.sin(2 * np.pi * np.arange(365) / 7)  # Weekly seasonality
noise = np.random.normal(0, 2, 365)
values = 100 + trend + seasonality + noise

df = pd.DataFrame({
    'ds': dates,
    'y': values
})

print("Data shape:", df.shape)
print("\nFirst few rows:")
print(df.head())

# Split into train and test
train = df[df['ds'] < '2020-11-01']
test = df[df['ds'] >= '2020-11-01']

print(f"\nTrain size: {len(train)}, Test size: {len(test)}")

# Create and fit model with US holidays
model = UniversalForecaster(
    backend='prophet',
    country_holidays=['US']
)

print("\nFitting model...")
model.fit(train)

# Generate forecasts
print("Generating forecasts...")
forecast = model.predict(horizon=len(test))

print("\nForecast shape:", forecast.shape)
print("\nFirst few forecast rows:")
print(forecast.head())

# Evaluate
results = evaluate(
    test,
    forecast,
    metrics=['mae', 'rmse', 'mape']
)

print("\nEvaluation Results:")
print(results)

# Show model info
print("\nModel Info:")
info = model.get_model_info()
for key, value in info.items():
    if key != 'backend_info':
        print(f"  {key}: {value}")


In [1]:
"""Quickstart example using AutoGluon backend with panel data."""

import pandas as pd
import numpy as np
from universal_ts import UniversalForecaster, evaluate

# Create synthetic panel data (multiple stores)
np.random.seed(42)
n_stores = 3
n_days = 180
data = []

for store_id in range(n_stores):
    dates = pd.date_range('2020-01-01', periods=n_days, freq='D')
    
    # Each store has different trend and base level
    base = 100 + store_id * 50
    trend = np.arange(n_days) * (0.2 + store_id * 0.1)
    seasonality = 15 * np.sin(2 * np.pi * np.arange(n_days) / 7)
    noise = np.random.normal(0, 5, n_days)
    values = base + trend + seasonality + noise
    
    store_df = pd.DataFrame({
        'store_id': f'store_{store_id}',
        'ds': dates,
        'y': values
    })
    data.append(store_df)

df = pd.concat(data, ignore_index=True)

print("Panel data shape:", df.shape)
print(f"Number of stores: {df['store_id'].nunique()}")
print("\nFirst few rows:")
print(df.head())

# Split into train and test
train = df[df['ds'] < '2020-05-01']
test = df[df['ds'] >= '2020-05-01']

print(f"\nTrain size: {len(train)}, Test size: {len(test)}")

# Create and fit AutoGluon model
# GPU is automatically detected and used if available.
# You can also explicitly set num_gpus=1 or num_gpus=0 to force GPU/CPU.
model = UniversalForecaster(
    backend='autogluon',
    prediction_length=30,
    eval_metric='MASE',
    verbosity=2,
    num_gpus=None  # Set to 1 if you want to force GPU usage
)

print("\nFitting AutoGluon model (this may take a minute)...")
model.fit(train, group_id='store_id', time_limit=60)

# Generate forecasts
print("\nGenerating forecasts...")
forecast = model.predict(horizon=30)

print("\nForecast shape:", forecast.shape)
print("\nSample forecasts per store:")
for store in df['store_id'].unique():
    store_forecast = forecast[forecast['store_id'] == store]
    print(f"\n{store}: {len(store_forecast)} forecasts")
    print(store_forecast.head(3))

# Evaluate per store and overall
results = evaluate(
    test,
    forecast,
    metrics=['mae', 'rmse', 'mape'],
    group_id_col='store_id'
)

print("\nEvaluation Results (per store + overall):")
print(results)

# Show model info
print("\nModel Info:")
info = model.get_model_info()
for key, value in info.items():
    if key not in ['backend_info']:
        print(f"  {key}: {value}")


Panel data shape: (540, 3)
Number of stores: 3

First few rows:
  store_id         ds           y
0  store_0 2020-01-01  102.483571
1  store_0 2020-01-02  111.236151
2  store_0 2020-01-03  118.262361
3  store_0 2020-01-04  114.723405
4  store_0 2020-01-05   93.120977

Train size: 363, Test size: 177


c:\Users\gujar\.conda\envs\ts\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\gujar\.conda\envs\ts\Lib\site-packages\torch\cuda\__init__.py:235: UserWarning: 
NVIDIA GeForce RTX 5080 with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA GeForce RTX 5080 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(
c:\more ML\TimeSeriesLibrary\universal_ts\backends\autogluon_backend.py:97: UserWarning: GPU NVIDIA GeForce RTX 5080 with capability sm_120 is not compatible with current PyTorch installation (supported arches: ['sm_50', 'sm_60', 'sm_61', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90']). A

[autogluon] No GPU detected. Using CPU only.

Fitting AutoGluon model (this may take a minute)...


Models that will be trained: ['SeasonalNaive', 'RecursiveTabular', 'DirectTabular', 'DynamicOptimizedTheta', 'Chronos2', 'Chronos2SmallFineTuned', 'AutoETS', 'ChronosWithRegressor[bolt_small]', 'TemporalFusionTransformer', 'DeepAR']
Training timeseries model SeasonalNaive. Training for up to 5.4s of the 59.6s of remaining time.
	-1.3677       = Validation score (-MASE)
	0.01    s     = Training runtime
	2.58    s     = Validation (prediction) runtime
Training timeseries model RecursiveTabular. Training for up to 5.7s of the 57.0s of remaining time.
	-0.7483       = Validation score (-MASE)
	0.19    s     = Training runtime
	0.06    s     = Validation (prediction) runtime
Training timeseries model DirectTabular. Training for up to 6.3s of the 56.8s of remaining time.
	-2.1004       = Validation score (-MASE)
	0.22    s     = Training runtime
	0.03    s     = Validation (prediction) runtime
Training timeseries model DynamicOptimizedTheta. Training for up to 7.1s of the 56.5s of remaining


Generating forecasts...

Forecast shape: (90, 14)

Sample forecasts per store:

store_0: 30 forecasts
  store_id         ds        yhat         0.1         0.2         0.3  \
0  store_0 2020-05-01  137.740095  131.566111  133.685513  135.213751   
1  store_0 2020-05-02  132.071888  125.654511  127.857465  129.445949   
2  store_0 2020-05-03  117.425195  110.821019  113.088097  114.722820   

          0.4         0.5         0.6         0.7         0.8         0.9  \
0  136.519574  137.740095  138.960616  140.266439  141.794677  143.914078   
1  130.803251  132.071888  133.340525  134.697826  136.286311  138.489265   
2  116.119630  117.425195  118.730760  120.127571  121.762293  124.029371   

   yhat_lower  yhat_upper  
0  131.566111  143.914078  
1  125.654511  138.489265  
2  110.821019  124.029371  

store_1: 30 forecasts
   store_id         ds        yhat         0.1         0.2         0.3  \
30  store_1 2020-05-01  201.522890  194.427091  196.862933  198.619347   
31  store_1 